In [8]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# 1. KONFIGURASI PATH DAN PARAMETER (SESUAIKAN DENGAN LINGKUNGAN BAPAK)
# ==============================================================================
# Path ke model Keras 1C Bapak (.h5 atau folder SavedModel standar)
MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20" 

# Path ke file JSON STEAD 1C
KEY_TEST_FILE = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json'
SAVE_DIR = "Fase2_DomainAdaptation_Results"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

CALIBRATION_SIZE = 1000 # Jumlah N-samples untuk Few-Shot
TSSS_THRESHOLD = 0.70   # Ambang batas TSSS (90% keyakinan)
LEARNING_RATE = 1e-4    # Sangat kecil agar tidak merusak ingatan model global
EPOCHS = 5

# ==============================================================================
# 2. MODUL DTFFDA & NORMALISASI TENSOR
# ==============================================================================
def bandpass_filter(data, lowcut=1.0, highcut=25.0, fs=100.0, order=4):
    """
    Membuang anomali noise laut frekuensi sangat rendah (< 1 Hz) 
    dan frekuensi tinggi yang tidak dapat ditransfer (non-transferable).
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def max_abs_normalize(data):
    """Normalisasi amplitudo absolut maksimum untuk menyeragamkan magnitudo."""
    max_val = np.max(np.abs(data))
    if max_val == 0:
        return data
    return data / max_val

# ==============================================================================
# 3. EKSTRAKSI DAN FILTERING DATA 1-KOMPONEN (Z-AXIS)
# ==============================================================================
print("[INFO] Memuat dan mengekstraksi dataset STEAD 1C...")
with open(KEY_TEST_FILE, 'r') as f:
    data_stead = json.load(f)

X_all, y_all = [], []

for trace_name, trace_data in tqdm(data_stead.items(), desc="Pemrosesan DTFFDA"):
    try:
        # Proses kelas GEMPA (Label 1)
        if 'Z' in trace_data and len(trace_data['Z']) >= 700:
            z_sig = np.array(trace_data['Z'][:700])
            z_filt = bandpass_filter(z_sig)
            z_norm = max_abs_normalize(z_filt)
            X_all.append(z_norm)
            y_all.append(1) 
        
        # Proses kelas NOISE (Label 0)
        if 'Z_noise' in trace_data and len(trace_data['Z_noise']) >= 700:
            z_noise = np.array(trace_data['Z_noise'][:700])
            z_filt = bandpass_filter(z_noise)
            z_norm = max_abs_normalize(z_filt)
            X_all.append(z_norm)
            y_all.append(0)
            
    except Exception as e:
        continue # Lewati jika ada data yang rusak (corrupt)

if len(X_all) == 0:
    raise ValueError("Gagal Total: Array input kosong. Pastikan Key 'Z' dan 'Z_noise' ada di dalam JSON.")

X_all = np.array(X_all)[..., np.newaxis] # Reshape ke (N, 700, 1) untuk 1D-CNN
y_all = np.array(y_all)

# Splitting Data
X_calib, y_calib = X_all[:CALIBRATION_SIZE], y_all[:CALIBRATION_SIZE]
X_test, y_test = X_all[CALIBRATION_SIZE:], y_all[CALIBRATION_SIZE:]

# ==============================================================================
# 4. PEMUATAN MODEL & PARTIAL UNFREEZING
# ==============================================================================
print("\n[INFO] Memuat Model dan Mengeksekusi Partial Unfreezing...")
# Menggunakan Keras load_model agar aman dari error tf.saved_model
model = keras.models.load_model(MODEL_PATH)

# Membekukan (Freeze) seluruh ekstraktor fitur kecuali 2 lapisan terakhir
for layer in model.layers[:-2]:
    layer.trainable = False
for layer in model.layers[-2:]:
    layer.trainable = True

# Kompilasi ulang lapisan yang dibuka (unfrozen)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ==============================================================================
# 5. CLASS-BALANCED TARGET SAMPLES SELECTIVE STRATEGY (CB-TSSS)
# ==============================================================================

print(f"\n[INFO] Menjalankan Class-Balanced TSSS dengan threshold {TSSS_THRESHOLD}...")
probs = model.predict(X_calib, verbose=0)
confidences = np.max(probs, axis=1)
pseudo_labels = np.argmax(probs, axis=1)

# PERBAIKAN MUTLAK: Wajib ada indeks  di bagian paling akhir np.where
idx_noise = np.where((pseudo_labels == 0) & (confidences >= TSSS_THRESHOLD))
idx_gempa = np.where((pseudo_labels == 1) & (confidences >= TSSS_THRESHOLD))

print(f"       -> Lolos TSSS awal | Noise: {len(idx_noise)} sampel, Gempa: {len(idx_gempa)} sampel.")

# Mencari jumlah minimum antara kedua kelas agar SEIMBANG (Balanced)
min_samples = min(len(idx_noise), len(idx_gempa))

# Jika terlalu sedikit sampel yang yakin, kita turunkan sedikit threshold-nya sementara
if min_samples < 5:
    print("[WARNING] Terlalu sedikit sampel Gempa/Noise yang meyakinkan. Menurunkan threshold TSSS sementara ke 0.70...")
    idx_noise = np.where((pseudo_labels == 0) & (confidences >= 0.70))
    idx_gempa = np.where((pseudo_labels == 1) & (confidences >= 0.70))
    min_samples = min(len(idx_noise), len(idx_gempa))

if min_samples == 0:
    raise RuntimeError("Model sama sekali tidak bisa menebak kelas Gempa/Noise dengan yakin. Filter Frekuensi (DTFFDA) mungkin terlalu kuat. Coba ubah bandpass_filter menjadi (1.0, 20.0).")

# Mengambil top-K sampel paling meyakinkan (agar rasio 50:50)
# Sortir berdasarkan confidence tertinggi
idx_noise_sorted = idx_noise[np.argsort(confidences[idx_noise])[::-1]][:min_samples]
idx_gempa_sorted = idx_gempa[np.argsort(confidences[idx_gempa])[::-1]][:min_samples]

# Gabungkan kembali
final_tsss_idx = np.concatenate([idx_noise_sorted, idx_gempa_sorted])
np.random.shuffle(final_tsss_idx) # Acak urutannya agar model tidak bias

X_tsss = X_calib[final_tsss_idx]
y_tsss = pseudo_labels[final_tsss_idx]

print(f"       -> Data kalibrasi diseimbangkan menjadi: {len(X_tsss)} sampel ({min_samples} Noise vs {min_samples} Gempa).")
# ==============================================================================
# 6. FEW-SHOT FINE-TUNING
# ==============================================================================
print("\n[INFO] Memulai Pelatihan Ulang (Fine-Tuning) pada lapisan terakhir...")
# Tambahkan class_weight untuk jaminan ekstra
class_weights = {0: 1.0, 1: 1.0} 
model.fit(X_tsss, y_tsss, epochs=EPOCHS, batch_size=16, class_weight=class_weights, verbose=1)

# ==============================================================================
# 7. EVALUASI PEMULIHAN SPESIFISITAS (TNR > 90%)
# ==============================================================================
print("\n[INFO] Mengeksekusi Uji Ketahanan pada Unseen Test Set...")
preds_test = np.argmax(model.predict(X_test, verbose=0), axis=1)

cm = confusion_matrix(y_test, preds_test)
tn, fp, fn, tp = cm.ravel()

accuracy = (tp + tn) / len(y_test)
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0 # Recall / Sensitivitas
tnr = tn / (tn + fp) if (tn + fp) > 0 else 0 # Spesifisitas
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0 # Presisi

print("\n" + "="*50)
print("HASIL EVALUASI FASE 2: ADAPTASI DOMAIN TSSS")
print("="*50)
print(f"Akurasi Total          : {accuracy*100:.2f}%")
print(f"Sensitivitas (TPR)     : {tpr*100:.2f}%")
print(f"Spesifisitas (TNR)     : {tnr*100:.2f}%  <-- TARGET BAPAK > 90%")
print(f"Presisi Gempa (PPV)    : {ppv*100:.2f}%")
print("="*50)

# ==============================================================================
# 8. PENYIMPANAN VISUALISASI
# ==============================================================================
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Noise (NO)', 'Gempa (LE)'],
            yticklabels=['Noise (NO)', 'Gempa (LE)'])
plt.title(f'Confusion Matrix (Fase 2: Partial Unfreeze 1C)\nTNR: {tnr*100:.2f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()

plt_path = os.path.join(SAVE_DIR, "fase2_tsss_confusion_matrix.png")
plt.savefig(plt_path, dpi=300)
print(f"[INFO] Plot Confusion Matrix tersimpan di: {plt_path}")


[INFO] Memuat dan mengekstraksi dataset STEAD 1C...


Pemrosesan DTFFDA: 100%|██████████| 5000/5000 [00:02<00:00, 2028.39it/s]



[INFO] Memuat Model dan Mengeksekusi Partial Unfreezing...



[INFO] Menjalankan Class-Balanced TSSS dengan threshold 0.7...
       -> Lolos TSSS awal | Noise: 1 sampel, Gempa: 1 sampel.
[WARNING] Terlalu sedikit sampel Gempa/Noise yang meyakinkan. Menurunkan threshold TSSS sementara ke 0.70...


TypeError: only integer scalar arrays can be converted to a scalar index